In [1]:
import pandas as pd

df = pd.read_csv('data/mehko_businesses.csv', dtype={'postal_code': str})
df.head()


,business_name,address,city,postal_code,business_type,cuisine_tags,description,instagram,website,social_confidence
0,3 UP BURGERS,532 SAN BENITO ST UNIT 7,LOS ANGELES,90033,Sandwiches/Burgers,"burgers, cheeseburgers, smash burgers",Serves burgers with various toppings in Los An...,NaN,NaN,low
1,ABSOLUTE SWEETNESS CAFE,L716 CARMELITA PL,MONTEBELLO,90640,Mexican,"mexican, cafe, sweets",Mom-and-pop Mexican cafe offering food and swe...,@absolute.sweetness,NaN,high
2,ADA DELICIOSA COMIDA GUAT,18646 BRYANT ST,NORTHRIDGE,91324,Salvadoran/Guatemalan,"guatemalan, comida tipica",Serves delicious homemade Guatemalan food in N...,NaN,NaN,low
3,ALEJANDRA'S COCINA,12512 PELLISSIER RD,WHITTIER,90601,Mexican,"mexican, carne asada, guisados","Mexican home kitchen serving carne asada, ques...",NaN,NaN,low
4,ALENA'S LAB KITCHEN,1647 W PACIFIC COAST HWY 87,WILMINGTON,90744,Other,"home kitchen, homemade",Home kitchen business in Wilmington with limit...,@alenaslabkitchen,https://www.alenaslabkitchen.com,high


In [2]:
import pandas as pd
from utils import geocode_census_batch

geo_raw = geocode_census_batch(df)

df_census = df.merge(
    geo_raw.rename(columns={"matched_addr": "matched_addr_census",
                             "lat": "lat_census", "lon": "lon_census"}),
    left_index=True, right_on="id", how="left"
)
print(f"\nCensus matched: {df_census['lat_census'].notna().sum()}/{len(df_census)}")


  Census: rows 0–269

Census matched: 260/270


In [3]:
# Write Census coordinates to the geocoded CSV (run once to switch from GDB → Census)
import pandas as pd

comp = pd.read_csv('data/geocode_comparison.csv', dtype={'postal_code': str})

geocoded = comp[comp['lat_census'].notna()][
    ['business_name', 'address', 'city', 'postal_code', 'lat_census', 'lon_census', 'matched_addr_census']
].rename(columns={'lat_census': 'lat', 'lon_census': 'lon', 'matched_addr_census': 'matched_address'})

geocoded.to_csv('data/mehko_businesses_geocoded.csv', index=False)
print(f"Saved Census coords → mehko_businesses_geocoded.csv  ({len(geocoded)} rows)")


Saved Census coords → mehko_businesses_geocoded.csv  (260 rows)


In [4]:
import time, json, os
import pandas as pd
from utils import search_business

df = pd.read_csv('data/mehko_businesses.csv', dtype={'postal_code': str})
SNIPPET_CACHE = 'data/business_snippets.json'

if os.path.exists(SNIPPET_CACHE):
    with open(SNIPPET_CACHE) as f:
        snippets = json.load(f)
    print(f"Loaded {len(snippets)} cached snippets")
else:
    snippets = {}
    for i, row in df.iterrows():
        snippets[row['business_name']] = search_business(row['business_name'], row['city'])
        if (i + 1) % 20 == 0:
            print(f"  Searched {i+1}/{len(df)}...")
        time.sleep(0.3)
    with open(SNIPPET_CACHE, 'w') as f:
        json.dump(snippets, f, indent=2)
    print(f"Saved {len(snippets)} snippets → {SNIPPET_CACHE}")

found = sum(1 for v in snippets.values() if v)
print(f"Snippets found: {found}/{len(snippets)} ({found/len(snippets):.0%})")


Loaded 270 cached snippets
Snippets found: 237/270 (88%)


In [5]:
import anthropic, json, os
import pandas as pd
from utils import classify_batch

CLASSIFIED_CACHE = 'data/business_classifications.json'

if os.path.exists(CLASSIFIED_CACHE):
    with open(CLASSIFIED_CACHE) as f:
        classifications = json.load(f)
    print(f"Loaded {len(classifications)} cached classifications")
else:
    client = anthropic.Anthropic()
    df = pd.read_csv('data/mehko_businesses.csv', dtype={'postal_code': str})
    BATCH_SIZE = 30
    classifications = {}
    for start in range(0, len(df), BATCH_SIZE):
        batch = df.iloc[start:start + BATCH_SIZE]
        results = classify_batch(batch, snippets, client)
        for (_, row), result in zip(batch.iterrows(), results):
            classifications[row['business_name']] = result
        print(f"  Classified {min(start + BATCH_SIZE, len(df))}/{len(df)}")
    with open(CLASSIFIED_CACHE, 'w') as f:
        json.dump(classifications, f, indent=2)
    print(f"Saved → {CLASSIFIED_CACHE}")

df = pd.read_csv('data/mehko_businesses.csv', dtype={'postal_code': str})
df['business_type'] = df['business_name'].map(lambda n: classifications.get(n, {}).get('business_type', ''))
df['cuisine_tags']  = df['business_name'].map(lambda n: ', '.join(classifications.get(n, {}).get('cuisine_tags', [])))
df['description']   = df['business_name'].map(lambda n: classifications.get(n, {}).get('description', ''))
df.to_csv('data/mehko_businesses.csv', index=False)
print("Updated mehko_businesses.csv")
df[['business_name', 'business_type', 'cuisine_tags', 'description']].head(10)


Loaded 270 cached classifications
Updated mehko_businesses.csv


,business_name,business_type,cuisine_tags,description
0,3 UP BURGERS,Sandwiches/Burgers,"burgers, cheeseburgers, smash burgers",Serves burgers with various toppings in Los An...
1,ABSOLUTE SWEETNESS CAFE,Mexican,"mexican, cafe, sweets",Mom-and-pop Mexican cafe offering food and swe...
2,ADA DELICIOSA COMIDA GUAT,Salvadoran/Guatemalan,"guatemalan, comida tipica",Serves delicious homemade Guatemalan food in N...
3,ALEJANDRA'S COCINA,Mexican,"mexican, carne asada, guisados","Mexican home kitchen serving carne asada, ques..."
4,ALENA'S LAB KITCHEN,Other,"home kitchen, homemade",Home kitchen business in Wilmington with limit...
5,ALLEN'S BALIWAG SPECIAL CHI,Filipino,"filipino, lechon, bbq","Filipino food featuring Baliwag lechon manok, ..."
6,ALMUERZO CASERO,Latin American,"latin, almuerzo, homestyle",Homestyle Latin American lunch meals served fr...
7,AMBER KITCHEN,Breakfast/Brunch,"brunch, coffee, lunch",Kitchen and coffee spot offering brunch and lu...
8,ANDRES S FERNANDEZ LLC,Sandwiches/Burgers,"grilled cheese, bar food, sandwiches",Operates as The 805 Bar & Grilled Cheese offer...
9,ANGEL CITY DUMPLING CO.,Other,"dumplings, fusion, shawarma","Fusion dumplings filled with chicken shawarma,..."


# Build Interactive Map\nFolium map with marker clusters + popups. Exported as a self-contained HTML file for GitHub Pages.

In [6]:
import pandas as pd
from utils import build_map

geo  = pd.read_csv('data/mehko_businesses_geocoded.csv', dtype={'postal_code': str})
info = pd.read_csv('data/mehko_businesses.csv',          dtype={'postal_code': str})

df = geo[geo['lat'].notna()].merge(info, on='business_name', how='left').fillna('')

businesses = [
    {
        'name':        r['business_name'],
        'type':        r['business_type'],
        'tags':        r['cuisine_tags'],
        'description': r['description'],
        'address':     f"{r['address_x']}, {r['city_x']} {r['postal_code_x']}",
        'instagram':   r.get('instagram', ''),
        'website':     r.get('website',   ''),
        'lat':         r['lat'],
        'lon':         r['lon'],
    }
    for _, r in df.iterrows()
]

cuisine_types = sorted(df['business_type'].dropna().unique())
build_map(businesses, cuisine_types)


Saved → index.html  (260 businesses)


In [8]:
import time, json, os
import anthropic
import pandas as pd
from utils import search_and_extract

df = pd.read_csv('data/mehko_businesses.csv', dtype={'postal_code': str})
client = anthropic.Anthropic()
SOCIAL_CACHE = 'data/business_social.json'

if os.path.exists(SOCIAL_CACHE):
    with open(SOCIAL_CACHE) as f:
        social = json.load(f)
    print(f"Loaded {len(social)} cached records")
else:
    social = {}
    for i, row in df.iterrows():
        ig, web, conf = search_and_extract(row['business_name'], row['city'], client)
        social[row['business_name']] = {'instagram': ig, 'website': web, 'confidence': conf}
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(df)} ...")
        time.sleep(0.3)
    with open(SOCIAL_CACHE, 'w') as f:
        json.dump(social, f, indent=2)
    print(f"Saved → {SOCIAL_CACHE}")

df['instagram']         = df['business_name'].map(lambda n: social.get(n, {}).get('instagram', ''))
df['website']           = df['business_name'].map(lambda n: social.get(n, {}).get('website', ''))
df['social_confidence'] = df['business_name'].map(lambda n: social.get(n, {}).get('confidence', ''))
df.to_csv('data/mehko_businesses.csv', index=False)

found_ig  = (df['instagram'] != '').sum()
found_web = (df['website']   != '').sum()
high_conf = (df['social_confidence'] == 'high').sum()
print(f"Instagram: {found_ig}/{len(df)}  |  Website: {found_web}/{len(df)}  |  High confidence: {high_conf}")
df#[df['instagram'] != ''][['business_name','instagram','website','social_confidence']].head(10)


Loaded 270 cached records
Instagram: 100/270  |  Website: 59/270  |  High confidence: 85


,business_name,address,city,postal_code,business_type,cuisine_tags,description,instagram,website,social_confidence
0,3 UP BURGERS,532 SAN BENITO ST UNIT 7,LOS ANGELES,90033,Sandwiches/Burgers,"burgers, cheeseburgers, smash burgers",Serves burgers with various toppings in Los An...,,,low
1,ABSOLUTE SWEETNESS CAFE,L716 CARMELITA PL,MONTEBELLO,90640,Mexican,"mexican, cafe, sweets",Mom-and-pop Mexican cafe offering food and swe...,@absolute.sweetness,,high
2,ADA DELICIOSA COMIDA GUAT,18646 BRYANT ST,NORTHRIDGE,91324,Salvadoran/Guatemalan,"guatemalan, comida tipica",Serves delicious homemade Guatemalan food in N...,,,low
3,ALEJANDRA'S COCINA,12512 PELLISSIER RD,WHITTIER,90601,Mexican,"mexican, carne asada, guisados","Mexican home kitchen serving carne asada, ques...",,,low
4,ALENA'S LAB KITCHEN,1647 W PACIFIC COAST HWY 87,WILMINGTON,90744,Other,"home kitchen, homemade",Home kitchen business in Wilmington with limit...,@alenaslabkitchen,https://www.alenaslabkitchen.com,high
...,...,...,...,...,...,...,...,...,...,...
265,WURST BARBEQUE,4639 MARWOOD DR,LOS ANGELES,90065,BBQ,"bbq, sausage, grilled",Barbecue home kitchen specializing in wurst-st...,@wurstbarbeque,,medium
266,YANA'S LADAS,8622 BEACH ST,LOS ANGELES,90002,Latin American,"empanadas, latin, homemade",Home kitchen serving ladas and Latin American-...,,,low
267,YANG'S FAMILY KITCHEN,24 SAN RAPHAEL PL,POMONA,91766,Chinese,"chinese, homestyle, family",Family-run home kitchen in Pomona serving home...,,,low
268,YUE YUMMY KITCHEN,1621 E ALASKA ST,WEST COVINA,91791,Chinese,"asian, chinese, soup",Asian home kitchen offering variety of dishes ...,,,low
